# Realistic dataset

In [1]:
from pathlib import Path
DATA_DIR = Path("../data")

In [2]:
def asbool(x):
    if isinstance(x, bool):
        return x
    elif isinstance(x, str):
        return x.lower() in ["true", "1", "yes"]
    elif isinstance(x, int):
        return x == 1
    else:
        return False

asbool(True), asbool("True"), asbool("1"), asbool(1), asbool("yes"), asbool(False), asbool("false"), asbool("0"), asbool(0), asbool("no")

(True, True, True, True, True, False, False, False, False, False)

In [ ]:
import zipfile
import json
from tqdm import tqdm

#zip_path = "../data/traces/PatchFinder_top10_Llama-3.3-70B-Instruct_traces.jsonl.zip"

def load_traces(zip_path):
    """
    Load traces from a zip file containing a single JSONL file.
    Returns a dictionary mapping traceId to a dictionary of startTimeUnixNano to call name.
    """
    
    traces = {}
    with zipfile.ZipFile(zip_path) as z:
        # Get the single file name inside the archive
        inner_name = z.namelist()[0]
        print(z.namelist())
        # Open the file for streaming
        with z.open(inner_name, "r") as f:
            for line in tqdm(f, total=5566212):
                # Decode bytes → str
                line = line.decode("utf-8").strip()
                if not line:
                    continue
                record = json.loads(line)
                span = record["resourceSpans"][0]["scopeSpans"][0]["spans"][0]
                
                call = span["name"]
                
                stats = traces.get(span["traceId"], {})
                stats[span["startTimeUnixNano"]] = call
                traces[span["traceId"]] = stats
    print(f"Loaded {len(traces)} traces from {zip_path}")
    return traces

In [ ]:
import pandas as pd

def load_steps_data(filepath, traces):
    """
    Load the steps data from a JSONL file and align it with the traces.
    Returns a dictionary mapping step number to a dictionary of call name to count, and a DataFrame of the original data.
    """
    with open(filepath, 'r') as f:
        data = []
        for line in tqdm(f):
            data.append(json.loads(line))
        df = pd.DataFrame(data)
    
        results = {}
        for idx, example in tqdm(df.iterrows(), total=len(df)):
            traceid = example["metadata"]["trace_id"]
            trace = traces[traceid]
            times = list(trace.keys())
            times.sort()
            
            step = -1
            skip = False
            previous = None
            for time in times:
                step_call = trace[time]
                if step_call == "CodeAgent.run":
                    skip = False
                    previous = None
                    continue
                if step_call == "OpenAIServerModel.generate":
                    previous = None
                    if not skip:
                        step += 1
                        skip = True
                    else:
                        skip = False
                    continue
                if previous == step_call:
                    continue
                else:
                    previous = step_call
                stats = results.get(step, {})
                stats[step_call] = stats.get(step_call, 0) + 1
                results[step] = stats
                skip = False
                
        return results, df

In [58]:

jsonl_files = [
 'PatchFinder_top10_Qwen3-235B-A22B-Instruct-2507.jsonl',
 'PatchFinder_top10_Llama-3.3-70B-Instruct.jsonl',
 'PatchFinder_top10_gemma-3-27b-it-new-clean.jsonl'
]

data = {}

for file in jsonl_files:
    subset = "_".join(file.split("_")[0:2])
    model = file.split("_")[-1].replace(".jsonl", "")
    print(subset, model)
        
    results_path = DATA_DIR / "output" / f"{subset}_{model}.jsonl"
    trace_path = DATA_DIR / "traces" / f"{subset}_{model}_traces.jsonl.zip"

    traces = load_traces(trace_path)
    
    with open(results_path, 'r') as f:
        output_data = []
        for line in tqdm(f):
            output_data.append(json.loads(line))
        df = pd.DataFrame(output_data)
    
        for idx, row in df.iterrows():
            trace_id = row["metadata"]["trace_id"]
            if "CWEReportTool" in traces[trace_id].values():
                
                data[(subset, model)] = data.get((subset, model), [])
                data[(subset, model)].append({
                        "cve_id": row["input"]["cve"],
                        "commit_id": row["input"]["commit_id"]
                    })

PatchFinder_top10 Qwen3-235B-A22B-Instruct-2507
['PatchFinder_top10_Qwen3-235B-A22B-Instruct-2507_traces.jsonl']


100%|██████████| 5566212/5566212 [03:41<00:00, 25146.75it/s]


Loaded 12456 traces from ../data/traces/PatchFinder_top10_Qwen3-235B-A22B-Instruct-2507_traces.jsonl.zip


12366it [00:00, 27660.36it/s]


PatchFinder_top10 Llama-3.3-70B-Instruct
['PatchFinder_top10_Llama-3.3-70B-Instruct_traces.jsonl']


  2%|▏         | 97175/5566212 [00:28<26:56, 3382.47it/s]


... (2 outputs left out)

PatchFinder_top10 gemma-3-27b-it-new-clean
['PatchFinder_top10_gemma-3-27b-it_traces.jsonl']


  3%|▎         | 146086/5566212 [01:16<47:07, 1916.95it/s]  


Loaded 12222 traces from ../data/traces/PatchFinder_top10_gemma-3-27b-it-new-clean_traces.jsonl.zip


12222it [00:00, 23897.04it/s]


In [ ]:
from datasets import load_dataset

ds = load_dataset("andstor/cvevc_candidates", "PatchFinder_top10")

In [98]:
for key in data.keys():
    print(key, len(data[key]))

    lookup = [(d["cve_id"], d["commit_id"]) for d in data[key]]

    ablation_ds = ds["test"].filter(lambda x: (x["cve"], x["commit_id"]) in lookup)

    name = str(key[1])
    if name == "gemma-3-27b-it-new-clean":
        name = "gemma-3-27b-it"

    ablation_ds.push_to_hub("andstor/cvevc_candidates", config_name=f"PatchFinder_top10_CWEReportTool_{name}", private=False, max_shard_size="250MB")

('PatchFinder_top10', 'Qwen3-235B-A22B-Instruct-2507') 3256




Creating parquet from Arrow format: 100%|██████████| 5/5 [00:00<00:00, 30.25ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :   5%|▌         |  561kB / 11.2MB,  702kB/s  


Processing Files (0 / 1)                :  15%|█▌        | 1.68MB / 11.2MB, 1.20MB/s  

Processing Files (0 / 1)                :  25%|██▌       | 2.81MB / 11.2MB, 1.56MB/s  
Processing Files (0 / 1)                :  35%|███▌      | 3.93MB / 11.2MB, 1.96MB/s  
Processing Files (0 / 1)                :  45%|████▌     | 5.05MB / 11.2MB, 2.30MB/s  
Processing Files (0 / 1)                :  60%|██████    | 6.74MB / 11.2MB, 2.81MB/s  


Processing Files (0 / 1)                :  70%|███████   | 7.86MB / 11.2MB, 2.62MB/s  
Processing Files (0 / 1)                :  75%|███████▌  | 8.42MB / 11.2MB, 2.63MB/s  
Processing Files (0 / 1)                :  85%|████████▌ | 9.54MB / 11.2MB, 2.81MB/s  
Processing Files (0 / 1)                :  9

('PatchFinder_top10', 'Llama-3.3-70B-Instruct') 1183



Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 34.31ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :  84%|████████▍ | 1.60MB / 1.90MB, 8.06MB/s  



Processing Files (1 / 1)                : 100%|██████████| 1.90MB / 1.90MB, 1.90MB/s  

Processing Files (1 / 1)                : 100%|██████████| 1.90MB / 1.90MB, 1.58MB/s  
New Data Upload                         : 100%|██████████| 1.90MB / 1.90MB, 1.58MB/s  
                                        : 100%|██████████| 1.90MB / 1.90MB            
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.29s/ shards]


('PatchFinder_top10', 'gemma-3-27b-it-new-clean') 19



Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 337.03ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (1 / 1)                : 100%|██████████| 63.2kB / 63.2kB,  315kB/s  

Processing Files (1 / 1)                : 100%|██████████| 63.2kB / 63.2kB,  158kB/s  
New Data Upload                         : 100%|██████████| 63.2kB / 63.2kB,  158kB/s  
                                        : 100%|██████████| 63.2kB / 63.2kB            
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.40s/ shards]
